# Семинар: «Работа с файлами данных»

Ноутбук основан на материалах лекции по работе с файлами, файловой системе, кодировкам, бинарным представлениям, сериализации, JSON, XML и парсингу XML в Python. В лекции отдельно рассматриваются:
- работа с файловой системой через `os` / `os.path`;
- текстовые файлы, кодировки ASCII / CP1251 / Unicode / UTF-8 / UTF-16;
- работа с байтами и бинарными файлами;
- сериализация в `pickle`;
- обмен данными в `JSON`;
- `XML` и парсинг XML через `BeautifulSoup`;
- получение JSON по HTTP через `requests`.

Перед началом обязательно задайте `STUDENT_ID` в ячейке ниже: от него зависит генерация данных и ответы в проверках.

## Правила
- Не меняй генератор данных.
- Используй функции.
- Для путей используй `Path` или `os.path`.
- Для чтения бинарных файлов не используй внешние библиотеки.
- Для XML используй `BeautifulSoup(..., "xml")`.
- Для HTTP используй `requests`.
- Все созданные тобой файлы должны записываться внутрь `LAB_ROOT`.
- В нескольких задачах требуется создать результирующие файлы.

## Структура
- Первая пара: задания 1–10.
- Вторая пара: задания 11–20.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import shutil
import socket
import threading
from collections import Counter, defaultdict
from functools import reduce
from itertools import islice
from pathlib import Path
from statistics import mean, median
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler

import requests
from bs4 import BeautifulSoup

In [ ]:
# Впишите свой идентификатор свой идентификатор.
STUDENT_ID = "123456"  # например: "123456" из "123456@edu.fa.ru"

def seed_from_student_code(student_code: str) -> int:
    digest = hashlib.sha256(student_code.encode("utf-8")).hexdigest()
    return int(digest[:16], 16)

SEED = seed_from_student_code(STUDENT_ID)
SEED

10202516757047595970

In [ ]:
RUS_WORDS = [
    "алгоритм", "данные", "модель", "файл", "кодировка", "строка", "символ", "байт",
    "объект", "сериализация", "десериализация", "путь", "директория", "словарь",
    "список", "запись", "поток", "архив", "отчёт", "проверка", "анализ", "формат",
    "таблица", "проект", "студент", "пакет", "контроль", "пример", "ресурс", "версия",
    "загрузка", "выгрузка", "запрос", "ответ", "документ", "атрибут", "элемент", "узел",
    "функция", "система", "сервер", "клиент", "облако", "сеть", "структура", "метаданные",
]
CITIES = [
    "Москва", "Казань", "Томск", "Пермь", "Омск", "Тула", "Самара",
    "Уфа", "Сочи", "Тверь", "Рязань", "Ижевск", "Калуга", "Вологда",
]
NAMES = [
    "Алина", "Борис", "Виктор", "Галина", "Денис", "Елена", "Жанна", "Игорь",
    "Кира", "Лев", "Марина", "Никита", "Ольга", "Павел", "Роман", "Светлана",
    "Тимур", "Ульяна", "Фёдор", "Юлия",
]
REGIONS = ["north", "south", "west", "east", "center"]
CATEGORIES = ["books", "hardware", "stationery", "media"]
PHONE_TYPES = ["work", "home", "personal"]

def safe_slug(value: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in value.lower())

def random_ru_sentence(rng: random.Random, min_words: int = 5, max_words: int = 10) -> str:
    words = rng.sample(RUS_WORDS, rng.randint(min_words, max_words))
    sent = " ".join(words)
    return sent.capitalize() + "."

def write_text(path: Path, text: str, encoding: str, newline: str | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding=encoding, newline=newline) as f:
        f.write(text)

def write_bytes(path: Path, data: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(data)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def generate_dataset(student_code: str, root_base: str = "lab03_generated") -> Path:
    rng = random.Random(seed_from_student_code(student_code))
    root = Path(root_base) / safe_slug(student_code)
    if root.exists():
        shutil.rmtree(root)
    root.mkdir(parents=True, exist_ok=True)

    # 1. Файловая система
    fs_root = root / "fs"
    fs_files = {
        "projects/alpha/readme.txt": "\n".join(random_ru_sentence(rng) for _ in range(3)),
        "projects/alpha/data/report.json": json.dumps(
            {"version": rng.randint(1, 5), "quality": round(rng.uniform(0.7, 0.99), 3), "owner": rng.choice(NAMES)},
            ensure_ascii=False,
            indent=2,
        ),
        "projects/beta/notes.md": "\n".join(f"- {random_ru_sentence(rng, 4, 7)}" for _ in range(5)),
        "projects/beta/tmp/ignore.log": "\n".join(random_ru_sentence(rng, 4, 6) for _ in range(2)),
        "archive/2024/summary.txt": "\n".join(random_ru_sentence(rng, 6, 9) for _ in range(4)),
        "archive/2024/raw/data.bin": "",
        "misc/config.yaml": "\n".join(
            [
                f"host: node-{rng.randint(1,9)}",
                f"port: {8000 + rng.randint(0,999)}",
                f"debug: {rng.choice(['true','false'])}",
            ]
        ),
        "misc/tmp/cache.txt": "\n".join(random_ru_sentence(rng, 3, 5) for _ in range(2)),
        "incoming/checkpoints/model.ckpt": "",
        "incoming/messages.txt": "\n".join(random_ru_sentence(rng, 5, 8) for _ in range(3)),
    }
    for rel_path, content in fs_files.items():
        path = fs_root / rel_path
        if rel_path.endswith(".bin"):
            write_bytes(path, bytes(rng.randint(0, 255) for _ in range(64)))
        elif rel_path.endswith(".ckpt"):
            write_bytes(path, bytes(rng.randint(0, 255) for _ in range(96)))
        else:
            write_text(path, content, encoding="utf-8")

    raw_paths = [
        "./projects/alpha/../alpha/readme.txt",
        ".\\projects\\alpha\\data\\..\\data\\report.json",
        "./archive/2024/./summary.txt",
        "misc/./config.yaml",
        "incoming/checkpoints/../messages.txt",
        "./projects/beta/tmp/../notes.md",
        ".\\archive\\2024\\raw\\..\\summary.txt",
    ]
    write_text(fs_root / "paths_raw.txt", "\n".join(raw_paths), encoding="utf-8")

    # 2. Текстовые файлы и кодировки
    texts_root = root / "texts"
    cp1251_lines = [random_ru_sentence(rng, 5, 9) for _ in range(6)]
    cp1251_lines[1] = cp1251_lines[1].replace(".", "") + " Ёжик ждёт ёлку."
    texts_root.mkdir(parents=True, exist_ok=True)
    with open(texts_root / "notes_cp1251.txt", "w", encoding="cp1251", newline="\r\n") as f:
        f.write("\n".join(cp1251_lines))

    utf8_sig_lines = [f"{city}: {random_ru_sentence(rng, 4, 7)}" for city in rng.sample(CITIES, 5)]
    write_text(texts_root / "poem_utf8_sig.txt", "\n".join(utf8_sig_lines), encoding="utf-8-sig")

    utf16_lines = [f"{i+1}. {name} — {rng.choice(CITIES)}" for i, name in enumerate(rng.sample(NAMES, 7))]
    write_text(texts_root / "names_utf16.txt", "\n".join(utf16_lines), encoding="utf-16")

    mixed_bytes = b"alpha\r\nbeta\ngamma\rdelta\r\nepsilon\nzeta\r"
    write_bytes(texts_root / "mixed_newlines.bin", mixed_bytes)

    # 3. Бинарные файлы
    bin_root = root / "binaries"
    numbers = [rng.randint(10_000, 999_999) for _ in range(25)]
    write_bytes(bin_root / "numbers_big.bin", b"".join(n.to_bytes(4, "big", signed=False) for n in numbers))

    sensor_records = []
    sensor_bytes = bytearray()
    for seq in range(1, 19):
        sensor_id = 1000 + rng.randint(1, 7)
        temp_x10 = rng.randint(-50, 350)
        hum_x10 = rng.randint(250, 990)
        status = rng.choice([0, 1, 1, 1, 2])
        battery = rng.randint(15, 100)
        sensor_records.append(
            {
                "sensor_id": sensor_id,
                "temp_x10": temp_x10,
                "hum_x10": hum_x10,
                "status": status,
                "battery": battery,
                "seq": seq,
            }
        )
        sensor_bytes.extend(sensor_id.to_bytes(4, "big", signed=False))
        sensor_bytes.extend(int(temp_x10).to_bytes(2, "big", signed=True))
        sensor_bytes.extend(int(hum_x10).to_bytes(2, "big", signed=False))
        sensor_bytes.extend(int(status).to_bytes(1, "big", signed=False))
        sensor_bytes.extend(int(battery).to_bytes(1, "big", signed=False))
        sensor_bytes.extend(int(seq).to_bytes(2, "big", signed=False))
    write_bytes(bin_root / "sensor_records.bin", bytes(sensor_bytes))

    packet_messages = [f"packet_{i}_{rng.choice(CITIES)}" for i in range(1, 11)]
    packets = []
    packets_bytes = bytearray()
    for msg in packet_messages:
        score = rng.randint(-200, 500)
        flag = rng.choice([0, 1])
        payload = msg.encode("utf-8")
        packets.append({"message": msg, "score": score, "flag": flag})
        packets_bytes.extend(len(payload).to_bytes(2, "big"))
        packets_bytes.extend(payload)
        packets_bytes.extend(int(score).to_bytes(4, "big", signed=True))
        packets_bytes.extend(int(flag).to_bytes(1, "big"))
    write_bytes(bin_root / "packets.bin", bytes(packets_bytes))

    # 4. Pickle
    pickle_root = root / "pickle_data"
    pickle_root.mkdir(parents=True, exist_ok=True)
    import pickle
    obj1 = [(rng.choice(NAMES), rng.randint(50, 100)) for _ in range(8)]
    obj2 = {
        "course": "data_files",
        "seed": seed_from_student_code(student_code),
        "threshold": rng.randint(60, 85),
        "cities": rng.sample(CITIES, 4),
    }
    with open(pickle_root / "objects.pickle", "wb") as f:
        pickle.dump(obj1, f)
        pickle.dump(obj2, f)

    # 5. JSON
    json_root = root / "json"
    orders = []
    contacts = []
    products = []
    for pid in range(1, 11):
        products.append(
            {
                "sku": f"SKU-{pid:03d}",
                "category": rng.choice(CATEGORIES),
                "base_price": round(rng.uniform(120, 2500), 2),
            }
        )

    for cid, name in enumerate(rng.sample(NAMES, 10), start=1):
        phones = []
        for _ in range(rng.randint(1, 3)):
            phones.append(
                {
                    "type": rng.choice(PHONE_TYPES),
                    "number": f"+7-9{rng.randint(10,99)}-{rng.randint(100,999)}-{rng.randint(10,99)}-{rng.randint(10,99)}",
                }
            )
        contacts.append(
            {
                "id": cid,
                "name": name,
                "city": rng.choice(CITIES),
                "email": f"{safe_slug(name)}{cid}@example.test",
                "phones": phones,
            }
        )

    for oid in range(1, 16):
        items = []
        for _ in range(rng.randint(1, 4)):
            product = rng.choice(products)
            items.append(
                {
                    "sku": product["sku"],
                    "category": product["category"],
                    "qty": rng.randint(1, 5),
                    "price": round(product["base_price"] * rng.uniform(0.85, 1.15), 2),
                }
            )
        orders.append(
            {
                "order_id": f"ORD-{oid:03d}",
                "region": rng.choice(REGIONS),
                "customer_id": rng.randint(1, 10),
                "delivered": rng.choice([True, True, False]),
                "items": items,
            }
        )

    write_text(json_root / "orders.json", json.dumps(orders, ensure_ascii=False, indent=2), encoding="utf-8")
    write_text(json_root / "contacts.json", json.dumps(contacts, ensure_ascii=False, indent=2), encoding="utf-8")
    write_text(json_root / "products.json", json.dumps(products, ensure_ascii=False, indent=2), encoding="utf-8")

    # 6. Локальный API (статические JSON-файлы)
    api_root = root / "api_data"
    warehouses = []
    shipments = []
    for wid in range(1, 6):
        warehouses.append(
            {
                "warehouse_id": f"W-{wid}",
                "city": rng.choice(CITIES),
                "capacity": rng.randint(150, 450),
                "current_load": rng.randint(20, 140),
            }
        )
        shipments.append(
            {
                "warehouse_id": f"W-{wid}",
                "incoming": rng.randint(5, 60),
                "outgoing": rng.randint(0, 40),
            }
        )
    write_text(api_root / "warehouses.json", json.dumps(warehouses, ensure_ascii=False, indent=2), encoding="utf-8")
    write_text(api_root / "shipments.json", json.dumps(shipments, ensure_ascii=False, indent=2), encoding="utf-8")

    # 7. XML
    xml_root = root / "xml"
    xml_contacts_parts = ['<?xml version="1.0" encoding="UTF-8"?>', "<contacts>"]
    for person in contacts:
        xml_contacts_parts.append(f'  <contact id="{person["id"]}" city="{person["city"]}">')
        xml_contacts_parts.append(f'    <name>{person["name"]}</name>')
        xml_contacts_parts.append(f'    <email>{person["email"]}</email>')
        xml_contacts_parts.append("    <phones>")
        for phone in person["phones"]:
            xml_contacts_parts.append(f'      <phone type="{phone["type"]}">{phone["number"]}</phone>')
        xml_contacts_parts.append("    </phones>")
        xml_contacts_parts.append("  </contact>")
    xml_contacts_parts.append("</contacts>")
    write_text(xml_root / "contacts.xml", "\n".join(xml_contacts_parts), encoding="utf-8")

    catalog_parts = ['<?xml version="1.0" encoding="UTF-8"?>', "<catalog>"]
    for product in products:
        qty = rng.randint(5, 50)
        country = rng.choice(["RU", "KZ", "BY", "AM"])
        catalog_parts.append(
            f'  <product sku="{product["sku"]}" category="{product["category"]}" country="{country}">'
        )
        catalog_parts.append(f'    <title>{product["sku"]} &amp; {product["category"]}</title>')
        catalog_parts.append(f'    <price>{product["base_price"]}</price>')
        catalog_parts.append(f'    <quantity>{qty}</quantity>')
        catalog_parts.append("  </product>")
    catalog_parts.append("</catalog>")
    write_text(xml_root / "catalog.xml", "\n".join(catalog_parts), encoding="utf-8")

    return root

LAB_ROOT = generate_dataset(STUDENT_ID)
LAB_ROOT

In [ ]:
def tree(path: Path, max_depth: int = 3) -> None:
    path = Path(path)
    root_depth = len(path.parts)
    for p in sorted(path.rglob("*")):
        depth = len(p.parts) - root_depth
        if depth > max_depth:
            continue
        indent = "    " * depth
        suffix = "/" if p.is_dir() else ""
        print(f"{indent}{p.name}{suffix}")

print(f"Корневая папка: {LAB_ROOT}")
tree(LAB_ROOT, max_depth=2)

In [ ]:
class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, format: str, *args) -> None:
        pass

def _find_free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def start_local_api_server(api_root: Path) -> tuple[ThreadingHTTPServer, threading.Thread, str]:
    api_root = Path(api_root).resolve()
    port = _find_free_port()

    class Handler(QuietHandler):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, directory=str(api_root), **kwargs)

    server = ThreadingHTTPServer(("127.0.0.1", port), Handler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    base_url = f"http://127.0.0.1:{port}"
    return server, thread, base_url

## Задания

### Задание 1. Рекурсивный обход файловой системы

Напиши функцию `task01_list_files_excluding_tmp(fs_root)`, которая рекурсивно обходит каталог `fs_root` и возвращает **отсортированный** список относительных путей ко всем файлам, **если в пути нет каталога `tmp`**.  
Используй `Path` или `os.walk`, не хардкодь имена файлов.

In [ ]:
def task01_list_files_excluding_tmp(fs_root: Path) -> list[str]:
    # YOUR CODE HERE

task01_answer = task01_list_files_excluding_tmp(LAB_ROOT / "fs")
task01_answer[:5]

### Задание 2. Статистика по расширениям

Напиши функцию `task02_extension_stats(fs_root)`, которая для файлов из задания 1 строит словарь вида  
`{расширение: {"count": ..., "bytes": ...}}`.  
Ключ для файлов **без расширения** задай как `"<no_ext>"`.

In [ ]:
def task02_extension_stats(fs_root: Path) -> dict:
    # YOUR CODE HERE

task02_answer = task02_extension_stats(LAB_ROOT / "fs")
task02_answer

### Задание 3. Нормализация путей

В файле `fs/paths_raw.txt` лежат пути с разными разделителями и сегментами `.` / `..`.  
Напиши функцию `task03_normalize_paths(path_file)`, которая возвращает **список уникальных нормализованных относительных путей** в формате POSIX (`/`). Порядок — лексикографический.

In [ ]:
def task03_normalize_paths(path_file: Path) -> list[str]:
    # YOUR CODE HERE

task03_answer = task03_normalize_paths(LAB_ROOT / "fs" / "paths_raw.txt")
task03_answer

### Задание 4. Чтение CP1251-текста

Напиши функцию `task04_cp1251_metrics(path)`, которая читает файл `texts/notes_cp1251.txt` в корректной кодировке и возвращает словарь:  
`{"line_count": ..., "nonempty_count": ..., "longest_line_len": ...}`.

In [ ]:
def task04_cp1251_metrics(path: Path) -> dict:
    # YOUR CODE HERE

task04_answer = task04_cp1251_metrics(LAB_ROOT / "texts" / "notes_cp1251.txt")
task04_answer

### Задание 5. Подсчёт типов переводов строк

Напиши функцию `task05_count_newlines(path)`, которая читает файл `texts/mixed_newlines.bin` **в бинарном режиме** и считает количество последовательностей:
- `windows` → `\r\n`
- `unix` → `\n`, не входящих в `\r\n`
- `mac` → `\r`, не входящих в `\r\n`

Верни словарь с этими тремя ключами.

In [ ]:
def task05_count_newlines(path: Path) -> dict:
    # YOUR CODE HERE

task05_answer = task05_count_newlines(LAB_ROOT / "texts" / "mixed_newlines.bin")
task05_answer

### Задание 6. Автоопределение кодировки

Напиши функцию `task06_detect_encoding(path)`, которая различает три случая:
- `utf-8-sig`
- `utf-16`
- `cp1251`

Затем построй словарь для трёх файлов:
- `notes_cp1251.txt`
- `poem_utf8_sig.txt`
- `names_utf16.txt`

Формат результата: `{имя_файла: {"encoding": ..., "first_nonempty_line": ...}}`.

In [ ]:
def task06_detect_encoding(path: Path) -> str:
    # YOUR CODE HERE

def task06_collect(texts_root: Path) -> dict:
    # YOUR CODE HERE

task06_answer = task06_collect(LAB_ROOT / "texts")
task06_answer

### Задание 7. Декодирование массива целых чисел

В файле `binaries/numbers_big.bin` записаны беззнаковые целые числа по 4 байта в формате big-endian.  
Напиши функцию `task07_decode_numbers(path)`, которая возвращает словарь:
`{"count": ..., "median": ..., "sum_last_digits": ...}`  
где `sum_last_digits` — сумма последних десятичных цифр всех чисел.

In [ ]:
def task07_decode_numbers(path: Path) -> dict:
    # YOUR CODE HERE

task07_answer = task07_decode_numbers(LAB_ROOT / "binaries" / "numbers_big.bin")
task07_answer

### Задание 8. Разбор бинарных записей датчиков

В файле `binaries/sensor_records.bin` каждая запись имеет длину 12 байт:
- `sensor_id` — 4 байта, big-endian, unsigned
- `temp_x10` — 2 байта, big-endian, signed
- `hum_x10` — 2 байта, big-endian, unsigned
- `status` — 1 байт
- `battery` — 1 байт
- `seq` — 2 байта, big-endian, unsigned

Напиши функцию `task08_sensor_summary(path)`, которая для записей со `status == 1` возвращает:
`{"active_count": ..., "top_sensor_by_avg_temp": ..., "avg_humidity": ...}`  
Температуру и влажность дели на 10.

In [ ]:
def task08_sensor_summary(path: Path) -> dict:
    # YOUR CODE HERE

task08_answer = task08_sensor_summary(LAB_ROOT / "binaries" / "sensor_records.bin")
task08_answer

### Задание 9. Разбор пакетов переменной длины

Файл `binaries/packets.bin` содержит последовательность пакетов:
- длина UTF-8 сообщения — 2 байта, unsigned, big-endian
- сообщение — UTF-8 bytes
- score — 4 байта, signed, big-endian
- flag — 1 байт

Напиши функцию `task09_decode_packets(path)`, которая возвращает список словарей только для пакетов, где `flag == 1` и `score > 0`.  
Каждый элемент: `{"message": ..., "score": ...}`.  
Список отсортируй по `score` по убыванию, затем по `message`.

In [ ]:
def task09_decode_packets(path: Path) -> list[dict]:
    # YOUR CODE HERE

task09_answer = task09_decode_packets(LAB_ROOT / "binaries" / "packets.bin")
task09_answer[:5]

### Задание 10. Последовательная загрузка Pickle-объектов

В файле `pickle_data/objects.pickle` последовательно сохранены **два** объекта pickle.  
Напиши функцию `task10_pickle_summary(path)`, которая:
1. загружает оба объекта;
2. считает число студентов с баллом не ниже порога `threshold` из второго объекта;
3. возвращает словарь  
`{"course": ..., "threshold": ..., "passed_count": ..., "cities_sorted": ...}`.

In [ ]:
def task10_pickle_summary(path: Path) -> dict:
    # YOUR CODE HERE

task10_answer = task10_pickle_summary(LAB_ROOT / "pickle_data" / "objects.pickle")
task10_answer

### Задание 11. Выручка по регионам из JSON

В `json/orders.json` лежит список заказов.  
Для каждого заказа сумма считается как `sum(qty * price)` по всем позициям.  
Напиши функцию `task11_revenue_by_region(path)`, которая возвращает словарь `{region: revenue}` **только для доставленных заказов**.  
Суммы округляй до 2 знаков.

In [ ]:
def task11_revenue_by_region(path: Path) -> dict:
    # YOUR CODE HERE

task11_answer = task11_revenue_by_region(LAB_ROOT / "json" / "orders.json")
task11_answer

### Задание 12. Лучший клиент

Напиши функцию `task12_best_customer(path)`, которая ищет клиента с максимальной выручкой по доставленным заказам.  
Верни словарь вида:
`{"customer_id": ..., "revenue": ..., "orders_count": ...}`  
При равенстве выручки выбирай меньший `customer_id`.

In [ ]:
def task12_best_customer(path: Path) -> dict:
    # YOUR CODE HERE

task12_answer = task12_best_customer(LAB_ROOT / "json" / "orders.json")
task12_answer

### Задание 13. Экспорт отчёта по категориям

Напиши функцию `task13_export_category_report(orders_path, out_path)`, которая:
1. строит отчёт по категориям товаров по **доставленным** заказам;
2. для каждой категории считает:
   - `qty_total`
   - `revenue_total`
3. сохраняет JSON-файл `out_path` с красивым форматированием и UTF-8.

Функция должна вернуть содержимое словаря отчёта.

In [ ]:
def task13_export_category_report(orders_path: Path, out_path: Path) -> dict:
    # YOUR CODE HERE

task13_out = LAB_ROOT / "results_task13.json"
task13_answer = task13_export_category_report(LAB_ROOT / "json" / "orders.json", task13_out)
task13_answer

### Задание 14. Локальный REST API через requests

Используя `requests`, получи данные из локального API.  
Доступны два ресурса:
- `/warehouses.json`
- `/shipments.json`

Сначала запусти сервер:
```python
server, thread, base_url = start_local_api_server(LAB_ROOT / "api_data")
```

Затем напиши функцию `task14_api_utilization(base_url)`, которая для каждого склада считает:
`new_load = current_load + incoming - outgoing`
и коэффициент загрузки `new_load / capacity`.

Верни словарь:
`{"max_utilized_warehouse": ..., "max_utilization": ...}`  
Коэффициент округли до 4 знаков.

In [ ]:
server, thread, base_url = start_local_api_server(LAB_ROOT / "api_data")

def task14_api_utilization(base_url: str) -> dict:
    # YOUR CODE HERE

task14_answer = task14_api_utilization(base_url)
task14_answer

### Задание 15. Разбор XML-контактов

Напиши функцию `task15_xml_contacts_summary(path)`, которая с помощью `BeautifulSoup(..., "xml")`:
1. извлекает все контакты;
2. считает число контактов;
3. находит контакт с максимальным числом телефонов;
4. возвращает словарь  
`{"contacts_count": ..., "top_name": ..., "top_phones": ...}`  
При равенстве выбирай имя, которое лексикографически меньше.

In [ ]:
def task15_xml_contacts_summary(path: Path) -> dict:
    # YOUR CODE HERE

task15_answer = task15_xml_contacts_summary(LAB_ROOT / "xml" / "contacts.xml")
task15_answer

### Задание 16. Стоимость каталога из XML

Напиши функцию `task16_catalog_value(path)`, которая из файла `xml/catalog.xml` извлекает все товары и возвращает словарь:
`{"products_count": ..., "max_sku": ..., "total_value": ...}`  
где:
- `products_count` — число товаров;
- `max_sku` — SKU товара с максимальной стоимостью партии `price * quantity`;
- `total_value` — суммарная стоимость всех партий, округлённая до 2 знаков.

In [ ]:
def task16_catalog_value(path: Path) -> dict:
    # YOUR CODE HERE

task16_answer = task16_catalog_value(LAB_ROOT / "xml" / "catalog.xml")
task16_answer

### Задание 17. Сверка JSON и XML представлений контактов

В файлах `json/contacts.json` и `xml/contacts.xml` содержатся одни и те же контакты в двух форматах.  
Напиши функцию `task17_compare_contacts(json_path, xml_path)`, которая проверяет совпадение для каждого `id`:
- `name`
- `email`
- число телефонов

Верни словарь:
`{"json_count": ..., "xml_count": ..., "mismatched_ids": [...]}`.

In [ ]:
def task17_compare_contacts(json_path: Path, xml_path: Path) -> dict:
    # YOUR CODE HERE

task17_answer = task17_compare_contacts(LAB_ROOT / "json" / "contacts.json", LAB_ROOT / "xml" / "contacts.xml")
task17_answer

### Задание 18. Плоское представление XML-каталога

Напиши функцию `task18_flatten_catalog(path)`, которая превращает `xml/catalog.xml` в список словарей вида:
`{"sku": ..., "category": ..., "country": ..., "title": ..., "lot_value": ...}`

`lot_value = price * quantity`.  
Список отсортируй по `lot_value` по убыванию, затем по `sku`.

In [ ]:
def task18_flatten_catalog(path: Path) -> list[dict]:
    # YOUR CODE HERE

task18_answer = task18_flatten_catalog(LAB_ROOT / "xml" / "catalog.xml")
task18_answer[:3]

### Задание 19. Манифест файлов

Напиши функцию `task19_manifest(root)`, которая для **всех файлов** внутри `root` строит список словарей:
`{"path": ..., "size": ..., "sha256_12": ...}`

где `sha256_12` — первые 12 символов SHA256-хэша файла.  
Список отсортируй по `path`.

In [ ]:
def task19_manifest(root: Path) -> list[dict]:
    # YOUR CODE HERE

task19_answer = task19_manifest(LAB_ROOT)
task19_answer[:5]

### Задание 20. Финальный отчёт и контрольная подпись

Собери итоговый отчёт `final_report.json` в корне `LAB_ROOT`.
Функцию изменять не требуется, она создает отчёт с результатами следующих задач:
- 3
- 8
- 11
- 14
- 17
- 19 (только число файлов `files_count`)

Только запусти функцию.  
Функция должна вернуть словарь:
`{"report_path": ..., "sha256": ...}`.

In [ ]:
def task20_build_final_report(root: Path) -> dict:
    report = {
        "student_code": STUDENT_ID,
        "task03": task03_answer,
        "task08": task08_answer,
        "task11": task11_answer,
        "task14": task14_answer,
        "task17": task17_answer,
        "files_count": len(task19_answer),
    }
    report_path = Path(root) / "final_report.json"
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    digest = sha256_file(report_path)
    sig_path = Path(root) / "final_report.sha256"
    sig_path.write_text(digest, encoding="utf-8")
    return {"report_path": report_path.as_posix(), "sha256": digest}

task20_answer = task20_build_final_report(LAB_ROOT)
task20_answer

## После выполнения
Проверь, что создан `final_report.json` и `final_report.sha256`.

## Завершение

In [ ]:
server.shutdown()